Name: José Alfredo Calvillo Gómez
Lab04

In [4]:
import sys
import os
import findspark

# 1. Inicializar findspark para que reconozca PySpark
findspark.init()

# 2. Configurar la ruta para encontrar tu clase Spark_utils
# Basado en tu salida anterior, la ruta correcta es:
ruta_modulo = "/opt/spark/work-dir/src/module"
if ruta_modulo not in sys.path:
    sys.path.append(ruta_modulo)

# 3. Importar PySpark y tu clase
try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import get_json_object, col
    from Spark_utils import SparkUtils # Importamos usando el nombre exacto del archivo
    print("¡Librerías y SparkUtils importados con éxito!")
except Exception as e:
    print(f"Error al importar: {e}")

# 4. Inicializar la sesión usando tu clase
utils = SparkUtils("Laboratorio 04")
spark = utils.get_spark_session()

¡Librerías y SparkUtils importados con éxito!


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 00:37:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
import os

# 1. Buscamos la ruta real de la carpeta data
# Probamos las rutas más probables según tu explorador de archivos
posibles_rutas = [
    "/opt/spark/work-dir/spark/data/",
    "/home/jovyan/work/spark/data/",
    "../../data/",
    "./spark/data/"
]

base_path = None
for ruta in posibles_rutas:
    if os.path.exists(ruta):
        base_path = ruta
        break

if base_path:
    print(f"Carpeta de datos encontrada en: {base_path}")
    
    # 2. Cargar catálogos
    df_agencies = spark.read.csv(f"{base_path}agencies.csv", header=True, inferSchema=True)
    df_cars = spark.read.csv(f"{base_path}cars.csv", header=True, inferSchema=True)
    df_customers = spark.read.csv(f"{base_path}customers.csv", header=True, inferSchema=True)

    # 3. Cargar Rentals (particionados)
    df_rentals = spark.read.csv(f"{base_path}part_*.csv", header=True, inferSchema=True)

    print("¡Datasets cargados con éxito!")
    df_rentals.show(1, truncate=False)
else:
    print("Error: No se pudo encontrar la carpeta /data/. Verifica que subiste los archivos.")

Carpeta de datos encontrada en: ../../data/
¡Datasets cargados con éxito!
+---------+-------------------------------------------------+
|rental_id|rental_info                                      |
+---------+-------------------------------------------------+
|11891    |{'car_id': 21, 'customer_id': 71, 'agency_id': 1}|
+---------+-------------------------------------------------+
only showing top 1 row


26/03/03 00:40:51 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../../data/part_*.csv.
java.io.FileNotFoundException: File ../../data/part_*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDa

In [8]:
from pyspark.sql.functions import get_json_object, col

# 1. Extraer nombre de la agencia
agencies_clean = df_agencies.select(
    "agency_id", 
    get_json_object(col("agency_info"), "$.agency_name").alias("agency_name")
)

# 2. Extraer nombre del cliente
customers_clean = df_customers.select(
    "customer_id", 
    get_json_object(col("customer_info"), "$.customer_name").alias("customer_name")
)

# 3. Extraer nombre del carro
cars_clean = df_cars.select(
    "car_id", 
    get_json_object(col("car_info"), "$.car_name").alias("car_name")
)

# 4. En Rentals, extraemos los IDs para poder hacer los joins después
# Los convertimos a 'int' para que coincidan con los IDs de las otras tablas
rentals_clean = df_rentals.select(
    "rental_id",
    get_json_object(col("rental_info"), "$.car_id").cast("int").alias("car_id"),
    get_json_object(col("rental_info"), "$.customer_id").cast("int").alias("customer_id"),
    get_json_object(col("rental_info"), "$.agency_id").cast("int").alias("agency_id")
)

print("Transformaciones listas (Lazy Evaluation aplicada).")

Transformaciones listas (Lazy Evaluation aplicada).


In [9]:
# Unimos todas las tablas usando sus IDs
final_df = rentals_clean \
    .join(cars_clean, "car_id") \
    .join(agencies_clean, "agency_id") \
    .join(customers_clean, "customer_id") \
    .select("rental_id", "car_name", "agency_name", "customer_name")

# LA ÚNICA ACCIÓN: Aquí es donde Spark realmente trabaja y muestra el resultado
final_df.show(truncate=False)

+---------+-----------------------------------+-------------+---------------+
|rental_id|car_name                           |agency_name  |customer_name  |
+---------+-----------------------------------+-------------+---------------+
|11891    |Wallace-Carlson Model 9            |NYC Rentals  |Margaret Jones |
|11892    |Grimes-Green Model 8               |LA Car Rental|Albert Williams|
|11893    |Stewart-Allen Model 5              |SF Cars      |Caleb Fleming  |
|11894    |Campos PLC Model 4                 |NYC Rentals  |Andrew Butler  |
|11895    |Wagner LLC Model 1                 |SF Cars      |Kristin Potts  |
|11896    |Jones, Jefferson and Rivera Model 7|LA Car Rental|Jeremy Parks   |
|11897    |Lopez and Sons Model 9             |Zapopan Auto |Terry Wells    |
|11898    |Salazar Ltd Model 8                |SF Cars      |Marc Williams  |
|11899    |Villanueva PLC Model 7             |LA Car Rental|Danny Williams |
|11900    |Faulkner-Howard Model 5            |SF Cars      |Eri